# Lesson 5: Tools And Routing

In [ ]:
import os
import datetime
import requests
import wikipedia
from pydantic import BaseModel, Field
from langchain.agents import tool
from langchain_anthropic import ChatAnthropic
from langchain.prompts import ChatPromptTemplate

from dotenv import load_dotenv

load_dotenv()
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

# Default model is Haiku 4.5; override with e.g. LLM_MODEL=claude-opus-4-8
MODEL = os.environ.get("LLM_MODEL", "claude-haiku-4-5")

In [38]:
@tool
def search(query: str) -> str:
    """search for weather online"""
    return "42f"


search.name

'search'

In [39]:
search.description

'search for weather online'

In [40]:
search.args

{'query': {'title': 'Query', 'type': 'string'}}

In [41]:
class SearchInput(BaseModel):
    query: str = Field(description="Thing to search for")

In [42]:
@tool(args_schema=SearchInput)
def search(query: str) -> str:
    """Search for the weather online."""
    return "42f"

In [43]:
search.args

{'query': {'description': 'Thing to search for',
  'title': 'Query',
  'type': 'string'}}

In [44]:
search.run("sf")

'42f'

In [45]:
class OpenMeteoInput(BaseModel):
    latitude: float = Field(..., description="Latitude of the location to fetch weather data for")
    longitude: float = Field(..., description="Longitude of the location to fetch weather data for")


@tool(args_schema=OpenMeteoInput)
def get_current_temperature(latitude: float, longitude: float) -> dict:
    """Fetch current temperature for given coordinates."""

    BASE_URL = "https://api.open-meteo.com/v1/forecast"

    # Parameters for the request
    params = {
        'latitude': latitude,
        'longitude': longitude,
        'hourly': 'temperature_2m',
        'forecast_days': 1,
    }

    response = requests.get(BASE_URL, params=params)

    if response.status_code == 200:
        results = response.json()
    else:
        raise Exception(f"API Request failed with status code: {response.status_code}")

    current_utc_time = datetime.datetime.now(datetime.timezone.utc)
    time_list = [datetime.datetime.fromisoformat(time_str).replace(tzinfo=datetime.timezone.utc) for time_str in
                 results['hourly']['time']]
    temperature_list = results['hourly']['temperature_2m']

    closest_time_index = min(range(len(time_list)), key=lambda i: abs(time_list[i] - current_utc_time))
    current_temperature = temperature_list[closest_time_index]

    return f'The current temperature is {current_temperature}°C'

In [46]:
get_current_temperature.name

'get_current_temperature'

In [47]:
get_current_temperature.description

'Fetch current temperature for given coordinates.'

In [48]:
get_current_temperature.args

{'latitude': {'description': 'Latitude of the location to fetch weather data for',
  'title': 'Latitude',
  'type': 'number'},
 'longitude': {'description': 'Longitude of the location to fetch weather data for',
  'title': 'Longitude',
  'type': 'number'}}

In [ ]:
from langchain_core.utils.function_calling import convert_to_openai_tool

convert_to_openai_tool(get_current_temperature)

get_current_temperature.invoke({"latitude": 13, "longitude": 14})

In [50]:
@tool
def search_wikipedia(query: str) -> str:
    """Run Wikipedia search and get page summaries."""
    page_titles = wikipedia.search(query)
    summaries = []
    for page_title in page_titles[: 3]:
        try:
            wiki_page = wikipedia.page(title=page_title, auto_suggest=False)
            summaries.append(f"Page: {page_title}\nSummary: {wiki_page.summary}")
        except (
                wikipedia.exceptions.PageError,
                wikipedia.exceptions.DisambiguationError,
        ):
            pass
    if not summaries:
        return "No good Wikipedia Search Result was found"
    return "\n\n".join(summaries)

In [51]:
search_wikipedia.name

'search_wikipedia'

In [52]:
search_wikipedia.description

'Run Wikipedia search and get page summaries.'

In [ ]:
convert_to_openai_tool(search_wikipedia)

In [ ]:
search_wikipedia.invoke({"query": "langchain"})

# Routing

In [ ]:
# bind_tools takes the @tool objects directly (provider-agnostic)
model = ChatAnthropic(model=MODEL, temperature=0).bind_tools(
    [search_wikipedia, get_current_temperature]
)

In [56]:
model.invoke("what is langchain")

AIMessage(content='', additional_kwargs={'function_call': {'arguments': '{"query":"Langchain"}', 'name': 'search_wikipedia'}}, response_metadata={'token_usage': {'completion_tokens': 16, 'prompt_tokens': 101, 'total_tokens': 117, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-3.5-turbo', 'system_fingerprint': None, 'finish_reason': 'function_call', 'logprobs': None}, id='run--aa587783-c79c-4ac8-8ce2-342a41ca4308-0')

In [57]:
model.invoke("what is rge weather in sf right now")

AIMessage(content='', additional_kwargs={'function_call': {'arguments': '{"latitude":37.7749,"longitude":-122.4194}', 'name': 'get_current_temperature'}}, response_metadata={'token_usage': {'completion_tokens': 25, 'prompt_tokens': 106, 'total_tokens': 131, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-3.5-turbo', 'system_fingerprint': None, 'finish_reason': 'function_call', 'logprobs': None}, id='run--1a91d882-5c51-46e4-be84-ef98990c0d93-0')

In [58]:
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are helpful but sassy assistant"),
    ("user", "{input}"),
])
chain = prompt | model

In [59]:
chain.invoke({"input": "what is the weather in sf right now"})


AIMessage(content='', additional_kwargs={'function_call': {'arguments': '{"latitude":37.7749,"longitude":-122.4194}', 'name': 'get_current_temperature'}}, response_metadata={'token_usage': {'completion_tokens': 25, 'prompt_tokens': 113, 'total_tokens': 138, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-3.5-turbo', 'system_fingerprint': None, 'finish_reason': 'function_call', 'logprobs': None}, id='run--bde78aea-7f77-478f-a188-27cf7f7ef58a-0')

In [ ]:
# The response is an AIMessage; tool calls (if any) are on .tool_calls
chain = prompt | model

In [ ]:
result = chain.invoke(({"input": "what is the weather in sf right now"}))

result.tool_calls

In [ ]:
get_current_temperature.invoke(result.tool_calls[0]["args"])
result = chain.invoke({"input": "hi"})

In [ ]:
# No tool call -> the answer is plain text on .content
result.content

In [ ]:
from langchain_core.messages import AIMessage


def route(result: AIMessage):
    if not result.tool_calls:
        # No tool call -> return the model's text answer
        return result.content
    tools = {
        "search_wikipedia": search_wikipedia,
        "get_current_temperature": get_current_temperature,
    }
    call = result.tool_calls[0]
    return tools[call["name"]].invoke(call["args"])

In [65]:
chain = prompt | model | OpenAIFunctionsAgentOutputParser() | route

In [66]:
result = chain.invoke({"input": "What is the weather in san francisco right now?"})
result

'The current temperature is 22.7°C'

In [67]:
result = chain.invoke({"input": "What is langchain?"})
result

'Page: LangChain\nSummary: LangChain is a software framework that helps facilitate the integration of large language models (LLMs) into applications. As a language model integration framework, LangChain\'s use-cases largely overlap with those of language models in general, including document analysis and summarization, chatbots, and code analysis.\n\n\n\nPage: Vector database\nSummary: A vector database, vector store or vector search engine is a database that uses the vector space model to store vectors (fixed-length lists of numbers) along with other data items. Vector databases typically implement one or more approximate nearest neighbor algorithms, so that one can search the database with a query vector to retrieve the closest matching database records.\nVectors are mathematical representations of data in a high-dimensional space. In this space, each dimension corresponds to a feature of the data, with the number of dimensions ranging from a few hundred to tens of thousands, dependi

In [68]:
chain.invoke({"input": "hi"})

'Well, hello there! How can I assist you today?'